# 🎯 DDoS Detection with Decision Tree
## SDN Spine-Leaf Network

## ⚠️ INSTALAR DEPENDENCIAS
```bash
pip3 install pandas numpy scikit-learn joblib matplotlib ipykernel --break-system-packages
```

---
# 📂 RUTA DEL DATASET - CAMBIAR ESTA RUTA

In [ ]:
DATASET_PATH = "/home/ryu/Desktop/Labo/SdnShare/datasets/dataset_20260301_075018.csv"

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

---
# 2. CARGAR Y LIMPIAR DATASET

In [ ]:
df = pd.read_csv(DATASET_PATH)
print(f"Dataset original: {df.shape[0]} filas, {df.shape[1]} columnas")

df = df.dropna()
print(f"Después de dropna: {df.shape[0]} filas")

numeric_cols = ['rx_pkts', 'tx_pkts', 'rx_bytes', 'tx_bytes', 'rx_pps', 'tx_pps', 'rx_bps', 'tx_bps']
df = df[~(df[numeric_cols] == 0).all(axis=1)]
print(f"Después de eliminar ceros: {df.shape[0]} filas")

df['label'] = df['label'].astype(int)

print(f"\nDistribución de labels:")
print(df['label'].value_counts())

---
# 3. PREPROCESAMIENTO

In [ ]:
feature_columns = [
    'rx_pkts', 'tx_pkts', 'rx_bytes', 'tx_bytes',
    'drops', 'errors', 'collisions',
    'rx_pps', 'tx_pps', 'rx_bps', 'tx_bps',
    'avg_packet_size_rx', 'avg_packet_size_tx', 'rx_tx_ratio'
]

X = df[feature_columns]
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]} muestras")
print(f"Test: {X_test.shape[0]} muestras")

---
# 4. ENTRENAR MODELO

In [ ]:
dt_model = DecisionTreeClassifier(
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)

dt_model.fit(X_train, y_train)

print("✅ Modelo entrenado")
print(f"Profundidad: {dt_model.get_depth()}")
print(f"Hojas: {dt_model.get_n_leaves()}")

---
# 5. EVALUACIÓN

In [ ]:
y_pred = dt_model.predict(X_test)

print("=" * 50)
print("📊 MÉTRICAS")
print("=" * 50)
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred):.4f}")

print("\n" + classification_report(y_test, y_pred, target_names=['Normal', 'Ataque']))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print("=" * 50)
print("🔢 MATRIZ DE CONFUSIÓN")
print("=" * 50)
print(f"              Predicho")
print(f"              Normal  Ataque")
print(f"Normal      {cm[0][0]:5d}  {cm[0][1]:5d}")
print(f"Ataque      {cm[1][0]:5d}  {cm[1][1]:5d}")

In [ ]:
feature_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': dt_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\n🎯 FEATURE IMPORTANCE:")
print(feature_importance.to_string(index=False))

---
# 6. VISUALIZACIÓN

In [ ]:
try:
    import matplotlib
    matplotlib.use('Agg')  # Usar backend sin GUI
    import matplotlib.pyplot as plt
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 1. Confusion Matrix
    im = axes[0].imshow(cm, cmap='Blues')
    axes[0].set_title('Matriz de Confusión', fontsize=12)
    axes[0].set_xticks([0, 1])
    axes[0].set_yticks([0, 1])
    axes[0].set_xticklabels(['Normal', 'Ataque'])
    axes[0].set_yticklabels(['Normal', 'Ataque'])
    axes[0].set_xlabel('Predicho')
    axes[0].set_ylabel('Real')
    
    for i in range(2):
        for j in range(2):
            axes[0].text(j, i, cm[i, j], ha='center', va='center', fontsize=14, fontweight='bold')
    
    plt.colorbar(im, ax=axes[0])
    
    # 2. Feature Importance
    axes[1].barh(feature_importance['feature'], feature_importance['importance'], color='steelblue')
    axes[1].set_xlabel('Importancia')
    axes[1].set_title('Feature Importance', fontsize=12)
    axes[1].invert_yaxis()
    
    plt.tight_layout()
    plt.savefig('/home/ryu/Desktop/Labo/SdnShare/models/model_results.png', dpi=150)
    print("✅ Gráficos guardados en: models/model_results.png")
    
except Exception as e:
    print(f"⚠️ Error al generar gráficos: {e}")
    print("Los gráficos se pueden ver en: models/model_results.png")

---
# 7. GUARDAR MODELO

In [ ]:
MODEL_PATH = "/home/ryu/Desktop/Labo/SdnShare/models/ddos_dt_model.pkl"
os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)

joblib.dump(dt_model, MODEL_PATH)

print(f"✅ Modelo guardado en: {MODEL_PATH}")

---
# 📊 RESUMEN FINAL

In [ ]:
print("=" * 60)
print("🎉 ENTRENAMIENTO COMPLETADO")
print("=" * 60)
print(f"• Dataset: {DATASET_PATH.split('/')[-1]}")
print(f"• Muestras totales: {df.shape[0]}")
print(f"• Muestras entrenamiento: {X_train.shape[0]}")
print(f"• Muestras test: {X_test.shape[0]}")
print(f"• Accuracy: {accuracy_score(y_test, y_pred):.2%}")
print(f"• Precision: {precision_score(y_test, y_pred):.2%}")
print(f"• Recall: {recall_score(y_test, y_pred):.2%}")
print(f"• F1-Score: {f1_score(y_test, y_pred):.2%}")
print(f"• Profundidad árbol: {dt_model.get_depth()}")
print(f"• Modelo: {MODEL_PATH}")
print("=" * 60)